# RedLine trên Kaggle — Ollama + Agent/LLM + giao diện chat

Chạy các cell theo thứ tự trong **phiên tương tác**. Trong Kaggle Settings bật **Internet** và chọn **GPU T4 x2** (hoặc GPU NVIDIA đủ VRAM).

Trong **Add-ons → Secrets**, tạo và Attach:
- `NGROK_AUTH_TOKEN`: bắt buộc để mở URL truy cập.
- `CANARY_TOKEN`: tùy chọn, dùng giá trị riêng nếu cần tái lập benchmark. Nếu bỏ trống, script sinh canary cho mỗi lần khởi động; không in secret ra notebook.

Mặc định chạy `qwen2.5:14b`. Có thể chọn `qwen2.5:7b` để giảm dung lượng tải/bộ nhớ. Ollama tự quyết định phân bổ GPU; có hai GPU không có nghĩa model luôn dùng cả hai.

URL public là sandbox dùng dữ liệu mock, gồm cả API quản lý tài liệu và guardrail không có xác thực. Chỉ chia sẻ với người tham gia thử nghiệm. Dịch vụ tồn tại trong phiên Kaggle; không phải hosting lâu dài.

**Chọn model:** mặc định `qwen2.5:14b`. Model nhẹ hơn khớp danh mục trong app: `qwen3.5:4b`, `qwen3:8b`, `llama3.1:8b` (đều hỗ trợ tool calling cho chế độ Agent). Đổi ở biến `MODEL` bên dưới.

**Guardrail & trace:** deployment Kaggle này chỉ chạy backend + Ollama. Hai chốt model **Llama Guard** và **Prompt Guard** là service Docker riêng nên **không có ở đây** — để bật/tắt chúng cần chạy bằng docker-compose trên máy (`make llama-guard-up` / `make prompt-guard-up`). Trên UI Kaggle, nếu bật nhầm chốt này thì request sẽ bị chặn (fail-closed). Các profile `none/basic/strict` (lọc regex) và **Trace** (xem suy luận + tool call) vẫn hoạt động bình thường.


In [ ]:
from pathlib import Path
import os, sys, json, time, signal, subprocess, shutil, urllib.request

REPO_URL = "https://github.com/TranQuangMinh-2005/RedLine.git"
BRANCH = "main"
MODEL = "qwen2.5:14b"
DEFENSE_PROFILE = "none"  # none / basic / strict
CUSTOMER_ID = "CUS-001"
ENABLE_UI = True         # False: chỉ mở FastAPI, không cài Node.js
CONTEXT_LENGTH = 8192
OLLAMA_VERSION = ""      # Rỗng: bộ cài hiện hành; điền version để tái lập
REPO_DIR = Path("/kaggle/working/RedLine")
RUNTIME_DIR = REPO_DIR / "runs/kaggle"

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Notebook này dành cho Kaggle. Chạy CLI trong README nếu dùng máy khác.")
if "launcher" in globals() and launcher.poll() is None:
    raise RuntimeError("Dịch vụ đang chạy. Chạy cell Dừng trước khi cấu hình lại.")
if DEFENSE_PROFILE not in {"none", "basic", "strict"}:
    raise ValueError("DEFENSE_PROFILE không hợp lệ")

from kaggle_secrets import UserSecretsClient
secrets_client = UserSecretsClient()
def read_secret(*names):
    for name in names:
        value = os.environ.get(name, "").strip()
        if value:
            return value
        try:
            value = secrets_client.get_secret(name).strip()
            if value:
                return value
        except Exception:
            pass
    return ""

ngrok_token = read_secret("NGROK_AUTH_TOKEN", "NGROK_AUTHTOKEN")
if not ngrok_token:
    raise RuntimeError("Thiếu NGROK_AUTH_TOKEN. Tạo Secret và bật Attach trước khi chạy.")
canary_token = read_secret("CANARY_TOKEN")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Cấu hình và Secrets: OK (không hiển thị giá trị secret).")

## 1. Lấy repo và cài môi trường Python riêng

Dùng Python 3.11 và `uv.lock`, không thay các package ML trong Python của Kaggle.
Update chỉ chấp nhận fast-forward; nếu repo có thay đổi local, cell dừng để bạn xử lý, không tự ghi đè.
Notebook gọi `scripts/kaggle_redline.py` trong repo, không chứa bản sao của launcher.

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
else:
    dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=REPO_DIR, text=True)
    if dirty.strip():
        raise RuntimeError("Repo có thay đổi chưa commit; lưu/xử lý chúng trước khi update.")
    subprocess.run(["git", "fetch", "origin", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "checkout", BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", f"origin/{BRANCH}"], cwd=REPO_DIR, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "uv"], check=True)
subprocess.run([sys.executable, "-m", "uv", "python", "install", "3.11"], check=True)
subprocess.run([sys.executable, "-m", "uv", "sync", "--locked", "--no-dev", "--python", "3.11"],
               cwd=REPO_DIR, check=True)
PYTHON = REPO_DIR / ".venv/bin/python"
subprocess.run([str(PYTHON), "--version"], check=True)
print("Commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip())

## 2. Cài Ollama và Node.js (nếu bật UI)

Bộ cài Ollama lấy từ nguồn chính thức. Node.js dùng bản 22 LTS và kiểm tra SHA256 của archive trước khi giải nén. Model được tải ở cell khởi động, vào `runs/kaggle/models` khi launcher tự chạy Ollama.

In [ ]:
import hashlib
import platform

RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
os.environ["PATH"] = "/usr/local/bin:/usr/bin:" + os.environ.get("PATH", "")
if not shutil.which("ollama") or OLLAMA_VERSION:
    sudo = [] if os.geteuid() == 0 else ["sudo"]
    if not shutil.which("zstd"):
        subprocess.run(sudo + ["apt-get", "update", "-qq"], check=True)
        subprocess.run(sudo + ["apt-get", "install", "-y", "-qq", "zstd"], check=True)
    installer = RUNTIME_DIR / "install-ollama.sh"
    urllib.request.urlretrieve("https://ollama.com/install.sh", installer)
    install_env = os.environ.copy()
    if OLLAMA_VERSION:
        install_env["OLLAMA_VERSION"] = OLLAMA_VERSION
    subprocess.run(["sh", str(installer)], env=install_env, check=True)

if ENABLE_UI:
    node_ok = False
    if shutil.which("node") and shutil.which("npm"):
        version = subprocess.check_output(["node", "--version"], text=True).strip()
        node_ok = int(version.lstrip("v").split(".")[0]) >= 20
    if not node_ok:
        if platform.machine() not in {"x86_64", "amd64"}:
            raise RuntimeError("Bộ cài Node của notebook yêu cầu Linux x86_64")
        base = "https://nodejs.org/dist/latest-v22.x/"
        with urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60) as response:
            checksums = response.read().decode()
        digest, filename = next(line.split() for line in checksums.splitlines()
                                if line.endswith("-linux-x64.tar.xz"))
        archive = RUNTIME_DIR / filename
        urllib.request.urlretrieve(base + filename, archive)
        with archive.open("rb") as stream:
            actual_digest = hashlib.file_digest(stream, "sha256").hexdigest()
        if actual_digest != digest:
            raise RuntimeError("Node archive checksum mismatch")
        node_dir = RUNTIME_DIR / "node"
        node_dir.mkdir(exist_ok=True)
        subprocess.run(["tar", "-xJf", str(archive), "--strip-components=1", "-C", str(node_dir)], check=True)
        os.environ["PATH"] = str(node_dir / "bin") + ":" + os.environ["PATH"]
        archive.unlink()
    subprocess.run(["node", "--version"], check=True)
print("Công cụ hệ thống: OK")

## 3. Khởi động và chờ kiểm thử

Launcher kiểm tra GPU/model, seed SQLite, ingest tài liệu, gọi thử LLM thuần và Agent, xác minh log có tool `search_knowledge` thành công. Sau đó build UI và mở tunnel.

Lần đầu có thể mất nhiều phút để tải model/build UI. Cell in log tiến độ, không in câu trả lời thử hay secret. Nếu bị lỗi hoặc bấm Interrupt, launcher được yêu cầu dừng và cleanup.
Khi báo `ready`, cell kết thúc nhưng dịch vụ tiếp tục chạy nền; dùng cell cuối để dừng.

In [ ]:
if "launcher" in globals() and launcher.poll() is None:
    raise RuntimeError("Launcher đang chạy. Dừng ở cell cuối trước khi chạy lại.")
launch_env = os.environ.copy()
launch_env["NGROK_AUTHTOKEN"] = ngrok_token
if canary_token:
    launch_env["CANARY_TOKEN"] = canary_token
else:
    launch_env.pop("CANARY_TOKEN", None)
launch_env["PYTHONUNBUFFERED"] = "1"
command = [str(PYTHON), "scripts/kaggle_redline.py", "--model", MODEL,
           "--defense-profile", DEFENSE_PROFILE, "--customer-id", CUSTOMER_ID,
           "--context-length", str(CONTEXT_LENGTH), "--runtime-dir", str(RUNTIME_DIR)]
if ENABLE_UI:
    command.append("--ui")
status_path = RUNTIME_DIR / "status.json"
# Preflight lock before removing status from a previous, already-stopped run.
import fcntl
with (RUNTIME_DIR / "launcher.lock").open("a") as check_lock:
    fcntl.flock(check_lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    status_path.unlink(missing_ok=True)
log_path = RUNTIME_DIR / "launcher.log"
with log_path.open("w") as log:
    launcher = subprocess.Popen(command, cwd=REPO_DIR, env=launch_env,
                                stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
try:
    deadline = time.monotonic() + 3600
    offset = 0
    while time.monotonic() < deadline:
        with log_path.open() as log:
            log.seek(offset)
            output = log.read()
            offset = log.tell()
        if output:
            print(output, end="", flush=True)
        if launcher.poll() is not None:
            raise RuntimeError(f"Launcher đã thoát ({launcher.returncode}). Xem log bên dưới.")
        if status_path.exists():
            status = json.loads(status_path.read_text())
            if status.get("state") == "ready":
                print("\nGiao diện:", status.get("ui_url") or "Không bật")
                print("API:", status["api_url"])
                print("OpenAI base URL:", status["api_url"] + "/v1")
                break
            if status.get("state") in {"failed", "stopped"}:
                raise RuntimeError(f"Startup {status['state']}; kiểm tra cell log.")
        time.sleep(2)
    else:
        raise TimeoutError("Quá 60 phút chờ khởi động")
except BaseException:
    if launcher.poll() is None:
        launcher.terminate()
        try:
            launcher.wait(timeout=90)
        except subprocess.TimeoutExpired:
            print("Cleanup còn chạy; xem log và dùng Stop session nếu cần.")
    raise

## 4. Kiểm tra API và xem tài liệu đã nạp

Dùng API nội bộ để kiểm tra nhanh, không qua trang ngrok. Cả hai chế độ đã được kiểm thử khi khởi động. Từ máy ngoài, dùng `status['api_url']` và header `ngrok-skip-browser-warning: true`.

Trong UI:
- **Agent / LLM thuần** (thanh bên trái): Agent có RAG + DB + tool; LLM thuần không gọi tool.
- **Model → Đổi**: chọn model/endpoint khi đang chạy (Kaggle này dùng Ollama local; tab OpenRouter/Groq/URL cần key tương ứng).
- **Trace** (header): bật để xem prompt đi qua các chốt guardrail, **chuỗi suy luận của model** và **tool nào được gọi kèm tham số + kết quả**.
- **defense none/basic/strict**: đổi lọc regex khi đang chạy; nút Llama Guard / Prompt Guard chỉ dùng khi các service đó được chạy (xem cell đầu).

API quản lý tài liệu ở `/rag/documents` (nhận JSON, chưa upload PDF/DOCX). Xem nội dung guardrail và cấu hình LLM qua `/config/guardrails`, `/config/llm`.

In [ ]:
def api_get(path):
    url = status["local_api_url"] + path
    with urllib.request.urlopen(url, timeout=15) as response:
        return json.load(response)

print(json.dumps(api_get("/health"), ensure_ascii=False, indent=2))
documents = api_get("/rag/documents")
print("Số tài liệu đã nạp:", documents["count"])
for document in documents["documents"]:
    print("-", document["document_id"], "—", document["title"])

## 5. Chẩn đoán khi có lỗi

- Thiếu GPU: bật accelerator rồi khởi động lại phiên.
- Thiếu Secret: bật Attach cho `NGROK_AUTH_TOKEN`.
- Hết VRAM/disk: dừng launcher, chọn model nhỏ hơn hoặc giảm context.
- Port bận: dừng lần chạy cũ; launcher không tự kill process không thuộc nó.
- Agent không gọi `search_knowledge`: kiểm tra model có tool calling, log và profile; launcher không coi một câu trả lời tự nhận “đã tra cứu” là đủ.
- Nếu reuse Ollama đã chạy trước đó, cấu hình daemon/model cache thuộc lần chạy đó. Stop cell chỉ dừng các process do launcher tạo.

Log/SQLite/model cache ở `runs/kaggle/` không được commit. Không công khai notebook output hoặc artifact chứa dữ liệu thử nhạy cảm.

In [ ]:
for filename in ("launcher.log", "ollama.log", "seed.log", "ingest.log", "target.log", "npm-install.log", "npm-build.log", "frontend.log"):
    path = RUNTIME_DIR / filename
    if path.exists():
        print(f"\n--- {filename} (30 dòng cuối) ---")
        print("\n".join(path.read_text(errors="replace").splitlines()[-30:]))

## 6. Dừng dịch vụ

Cell này dừng launcher, tunnel và các process do launcher tạo. DB/model cache vẫn giữ để chạy lại trong cùng phiên. Khi không dùng nữa, chọn **Stop session** trên Kaggle để ngừng phiên GPU.

In [ ]:
if "launcher" in globals() and launcher.poll() is None:
    launcher.terminate()
    try:
        launcher.wait(timeout=90)
        print("Đã dừng launcher và cleanup dịch vụ.")
    except subprocess.TimeoutExpired:
        print("Cleanup chưa hoàn tất. Xem log; dùng Stop session để dừng toàn bộ phiên Kaggle.")
else:
    print("Không có launcher đang chạy trong kernel này.")